## Requirements
- Serverless v4

## A. Setup

In [0]:
%pip install --upgrade databricks-langchain mlflow "unitycatalog-ai[databricks]==0.3.2" -qqq
%restart_python

In [0]:
catalog = "workspace"
schema = "bronze"
table_name = f"{catalog}.{schema}.sf_airbnb_listings"
avg_function_name = f"{catalog}.{schema}.avg_neigh_price"
cnt_function_name = f"{catalog}.{schema}.cnt_by_room_type"


# check if pre-requisite functions exist (see 01- notebook)
def table_exists(table_name):
    catalog, schema, tbl = table_name.split('.')
    result = spark.sql(f"SHOW TABLES IN {catalog}.{schema} LIKE '{tbl}'")
    return result.count() > 0

def function_exists(function_name):
    result = spark.sql(f"SHOW USER FUNCTIONS IN {catalog}.{schema} LIKE '{function_name.split('.')[-1]}'")
    return result.count() > 0

print(table_exists(table_name))
print(function_exists(avg_function_name))
print(function_exists(cnt_function_name))

In [0]:
df = spark.read.table(table_name)
display(df.limit(5))

## B. Logging the Agent

In [0]:
import mlflow


mlflow.langchain.autolog()
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_name = f"/Workspace/Users/{username}/single_agents_demo"

In [0]:
from importlib.metadata import version


user_query = "Get the average price for Mission and tell me the number of properties there that have a shared room"
# input_example = {"messages":[{"role": "user", "content": user_query}]}
input_example = {
    "input": [
        {"role": "user", "content": user_query}
    ]
}

model_name = "airbnb_knowledge_assistant"
tags_to_register = {
    "framework": "langchain",
    "stage": "dev",
    "version": "1"
}


# We are using config file. Otherwise, define the names and use the following:
# from mlflow.models.resources import (
#     DatabricksFunction,
#     DatabricksTable,
#     DatabricksServingEndpoint
# )
# resources = [
#     DatabricksFunction(function_name=function_name),
#     DatabricksTable(table_name=table_name), # redundant (?) as the functions already specify the target table names
#     DatabricksServingEndpoint(endpoint_name=endpoint_name)
# ]

In [0]:
# start an MLflow run and log the model
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent.py", # path to model from code
        code_paths=["agent-config.yaml"], # path to extra files
        input_example=input_example,
        pip_requirements=[
            f"unitycatalog-ai=={version('unitycatalog-ai')}",
            f"databricks-langchain=={version('databricks-langchain')}",
            f"langchain=={version('langchain')}",
            f"mlflow=={version('mlflow')}",
        ],
        # resources=resources # if resources are defined above
    )
    model_uri = logged_agent_info.model_uri

print(f"Model logged sucessfully with URI: {model_uri}")

## C. Reloading the Agent from Run and Validate I/O

In [0]:
output_example = mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/{model_name}",
    input_data=[input_example],
    env_manager="uv", # mlflow recommends using uv for better performance
)

## D. Registering the Agent

In [0]:
# Set the registry URI to UC:
mlflow.set_registry_uri("databricks-uc")

# Define the fully qualified model name in the UC
catalog = "workspace"
schema = "feature_model"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# Register the model
uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri,
    name=UC_MODEL_NAME
)

print(f"Model registered successfully to Unity Catalog.")
print(f"Model name: {UC_MODEL_NAME}")
print(f"Model version: {uc_registered_model_info.version}")